# Medical appointment data: exploratory data analysis\n\nThis EDA uses the original CSV. It documents quality, distributions, missingness, appointment outcomes, and calendar trends.\n

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
Path('artifacts').mkdir(exist_ok=True)
df = pd.read_csv(Path('Medical_appointment_data.csv'))
df.columns = df.columns.astype(str).str.strip().str.lower().str.replace('-', '_', regex=False)
removed_columns = [
    'average_rain_day', 'max_rain_day', 'diabetes', 'scholarship', 'alcoholism',
    'hipertension', 'patient_needs_companion', 'handcap', 'companion_child',
    'chronic_count', 'month_sin', 'appointment_shift', 'storm_day_before',
]
df = df.drop(columns=removed_columns, errors='ignore')
for column in df.select_dtypes(include=['object', 'string', 'category']).columns:
    df[column] = df[column].astype('string').str.strip().replace('', pd.NA)
df['appointment_date_continuous'] = pd.to_datetime(
    df['appointment_date_continuous'], errors='coerce', format='mixed', dayfirst=True
)
if df['appointment_date_continuous'].isna().any():
    raise ValueError('Appointment dates contain missing or invalid values.')
duplicate_record_count = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df['specialty'] = df['specialty'].fillna('General')
df['disability'] = df['disability'].fillna('No')
df.drop(columns=['under_12_years_old'], inplace=True, errors='ignore')
df.drop(columns=['over_60_years_old'], inplace=True, errors='ignore')
df.drop(columns=['patient_needs_companion'], inplace=True, errors='ignore')
df.drop(columns=['scholarship'], inplace=True, errors='ignore')
print(df.shape)
display(df.head())



## Missing-value handling

The specified source columns are excluded before imputation. Numeric values are filled with the mean when their distribution is approximately symmetric and the median when it is skewed; other categorical values use the mode. `specialty`, `disability`, and the outcome are not imputed.


In [ ]:
# Exclude the requested source columns from this notebook's analysis only.
excluded_columns = [
    'hipertension', 'alcoholism', 'handcap',
]
df = df.drop(columns=excluded_columns, errors='ignore')


# Keep these fields unchanged: specialty and disability may be meaningfully missing,
# and no_show is the outcome label rather than a predictor to impute.
protected_columns = {'specialty', 'disability', 'no_show'}
fill_summary = []

for column in df.select_dtypes(include='number').columns:
    if column in protected_columns or not df[column].isna().any():
        continue
    values = df[column].dropna()
    if values.empty:
        continue
    if abs(values.skew()) <= 1:
        method, fill_value = 'mean', values.mean()
    else:
        method, fill_value = 'median', values.median()
    missing_count = int(df[column].isna().sum())
    df[column] = df[column].fillna(fill_value)
    fill_summary.append({'column': column, 'method': method, 'values_filled': missing_count})

for column in df.select_dtypes(include=['object', 'string', 'category']).columns:
    if column in protected_columns or not df[column].isna().any():
        continue
    mode = df[column].mode(dropna=True)
    if mode.empty:
        continue
    missing_count = int(df[column].isna().sum())
    df[column] = df[column].fillna(mode.iloc[0])
    fill_summary.append({'column': column, 'method': 'mode', 'values_filled': missing_count})

fill_summary = pd.DataFrame(fill_summary, columns=['column', 'method', 'values_filled'])
print('Excluded columns:', ', '.join(excluded_columns))
if fill_summary.empty:
    print('No eligible missing values required imputation.')
else:
    display(fill_summary)


## Outlier removal

Invalid ages (below 0 or above 100) and IQR outliers in the selected continuous fields are removed before one-hot encoding. A row is removed when it falls outside the IQR bounds for any selected field. The original CSV remains unchanged.


In [ ]:
# Remove invalid ages and IQR outliers before encoding.
age_lower_bound = 0
age_upper_bound = 100
age_values = pd.to_numeric(df['age'], errors='coerce')
invalid_age_mask = age_values.notna() & ~age_values.between(age_lower_bound, age_upper_bound)
age_outlier_count = int(invalid_age_mask.sum())
df = df.loc[~invalid_age_mask].copy()

# Use the IQR rule for continuous fields only. A record is removed if it is an
# outlier in at least one of these fields.
outlier_columns = [
    column for column in [
        'age', 'appointment_time', 'avg_temp', 'average_temp_day',
        'max_temp', 'max_temp_day', 'avg_rain', 'max_rain',
    ] if column in df.columns
]
iqr_outlier_mask = pd.Series(False, index=df.index)
outlier_removal_summary = []

for column in outlier_columns:
    values = pd.to_numeric(df[column], errors='coerce')
    observed_values = values.dropna()
    if observed_values.empty:
        continue
    q1, q3 = observed_values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    column_outlier_mask = values.lt(lower_bound) | values.gt(upper_bound)
    iqr_outlier_mask |= column_outlier_mask.fillna(False)
    outlier_removal_summary.append({
        'feature': column,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound,
        'outlier_records_removed': int(column_outlier_mask.sum()),
    })

iqr_outlier_count = int(iqr_outlier_mask.sum())
df = df.loc[~iqr_outlier_mask].copy()
outlier_removal_summary = pd.DataFrame(outlier_removal_summary)
if not outlier_removal_summary.empty:
    outlier_removal_summary = outlier_removal_summary.sort_values(
        'outlier_records_removed', ascending=False
    ).reset_index(drop=True)

removal_overview = pd.DataFrame([{
    'invalid_age_records_removed': age_outlier_count,
    'iqr_outlier_rows_removed': iqr_outlier_count,
    'records_remaining': len(df),
}])
display(removal_overview)
display(outlier_removal_summary.round(2))

# Save the cleaned notebook data separately from the later encoded dataset.
output_dir = Path('artifacts')
output_dir.mkdir(exist_ok=True)
cleaned_output_path = output_dir / 'notebook_cleaned.csv'
df.to_csv(cleaned_output_path, index=False)
print(f'Saved cleaned data to: {cleaned_output_path}')


## One-hot encoding

Categorical predictors are converted to binary indicator columns. The target is not encoded.


In [ ]:
# One-hot encode categorical predictors and save the model-ready dataset.
# Place is deliberately excluded: it has thousands of distinct location values, which makes one-hot encoding impractically wide.
categorical_features = ['gender', 'specialty', 'disability', 'age_group', 'appointment_time_band', 'appointment_day_of_week', 'rain_intensity', 'heat_intensity']
categorical_features = [column for column in categorical_features if column in df.columns]
from time import perf_counter

start = perf_counter()
# Keep the target and redundant age flags out of the model-ready matrix.
encoding_excluded_columns = ['no_show']
X = df.drop(columns=encoding_excluded_columns, errors='ignore')
encoded_data = pd.get_dummies(X, columns=categorical_features, dummy_na=False, dtype='int8')
encoded_data['no_show'] = df['no_show'].map({'no': 0, 'yes': 1})
encoded_duplicate_count = int(encoded_data.duplicated().sum())
encoded_data = encoded_data.drop_duplicates().reset_index(drop=True)
constant_encoded_columns = encoded_data.columns[encoded_data.nunique(dropna=False).le(1)].tolist()
encoded_data = encoded_data.drop(columns=constant_encoded_columns)

# Save the encoded matrix as a separate exploratory artifact.
from datetime import datetime

output_dir = Path('artifacts')
output_dir.mkdir(exist_ok=True)
primary_output_path = output_dir / 'notebook_encoded.csv'
try:
    encoded_data.to_csv(primary_output_path, index=False)
    output_path = primary_output_path
except PermissionError:
    # The primary CSV is open elsewhere. Preserve this result in a new file instead.
    output_path = output_dir / f"notebook_encoded_{datetime.now():%Y%m%d_%H%M%S}.csv"
    encoded_data.to_csv(output_path, index=False)
    print(f'Could not overwrite {primary_output_path} because it is open. Saved the new encoded data to: {output_path}')
    print(f'To train with it now, run: python train_models.py --data {output_path}')

print('Excluded from encoded data:', ', '.join(encoding_excluded_columns))
print('Categorical features encoded:', categorical_features)
print(f'Removed {duplicate_record_count} exact source duplicates and {encoded_duplicate_count} duplicate feature rows.')
print('Removed constant encoded columns:', ', '.join(constant_encoded_columns) or 'none')
print(f'Encoded dataset: {encoded_data.shape[0]:,} rows x {encoded_data.shape[1]:,} columns')
print(f'Saved one-hot encoded data to: {output_path}')
print(f'Completed in {perf_counter() - start:.2f} seconds')
display(encoded_data.head())

# Review the fields saved to artifacts/notebook_encoded.csv.
# For numeric fields, the mean is also useful: for a one-hot field it is the
# proportion of appointments represented by that category.
numeric_means = encoded_data.mean(numeric_only=True).reindex(encoded_data.columns).round(4)
encoded_field_summary = pd.DataFrame({
    'field': encoded_data.columns,
    'data_type': encoded_data.dtypes.astype(str).to_numpy(),
    'example_values': [', '.join(map(str, encoded_data[column].dropna().unique()[:3]))
                       for column in encoded_data.columns],
    'mean': numeric_means.to_numpy(),
})
display(encoded_field_summary)


## Outlier-removal verification

This section verifies that no remaining record falls outside the IQR bounds used by the removal step. The boxplots show the distributions after cleaning.


In [ ]:
# Verify the rows kept for encoding against the IQR bounds used above.
verification_summary = []
for row in outlier_removal_summary.to_dict('records'):
    column = row['feature']
    values = pd.to_numeric(df[column], errors='coerce')
    remaining_outlier_count = int((values.lt(row['lower_bound']) | values.gt(row['upper_bound'])).sum())
    verification_summary.append({
        'feature': column,
        'remaining_outlier_count': remaining_outlier_count,
        'records_removed': row['outlier_records_removed'],
    })

verification_summary = pd.DataFrame(verification_summary).sort_values(
    'remaining_outlier_count', ascending=False
)
display(verification_summary)
if verification_summary['remaining_outlier_count'].sum() == 0:
    print('Verified: no retained rows fall outside the IQR bounds used for removal.')

# Use full-range whiskers for the cleaned-data plot. The usual 1.5-IQR
# boxplot rule would calculate new, tighter bounds after filtering and can mark
# valid retained rows as new fliers, even though they passed the removal rule.
fig, axes = plt.subplots(1, len(outlier_columns), figsize=(4 * len(outlier_columns), 4))
axes = [axes] if len(outlier_columns) == 1 else axes
for ax, column in zip(axes, outlier_columns):
    sns.boxplot(y=df[column], ax=ax, color='#72B7B2', whis=(0, 100), showfliers=False)
    ax.set_title(f'{column} (cleaned)')
    ax.set_xlabel('')
plt.tight_layout()
plt.show()


## Interpretation note\n\nThe charts are descriptive, not causal. Preserve missingness in later modeling and ensure all predictor values are known before the appointment.\n

## Duplicate-record removal\n
\n
Exact duplicate rows are removed from this notebook analysis and its encoded output. The original source CSV is not changed.\n

In [ ]:
# Duplicate removal is performed immediately after loading the source data.\n
print(f'Removed {duplicate_record_count} exact duplicate source records before cleaning and encoding.')

## Model explainability: SHAP and LIME

SHAP provides global and local Shapley-value explanations for the calibrated no-show probability. LIME gives a local, human-readable explanation for one selected appointment. Both sections use the saved classifier from `artifacts/classifier.joblib` and a reproducible sample of the encoded training data. Install the optional packages first if needed: `py -m pip install shap lime`.


In [ ]:
# SHAP: global feature effects and one local no-show explanation.
# Kernel SHAP requires numeric inputs. Categorical values are encoded only for
# SHAP and decoded before each prediction, so the final calibrated model still
# receives its original labelled input fields.
try:
    from pathlib import Path
    import joblib
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import shap
    from clinic_ml import CATEGORICAL, FEATURES, prepare_data
except ModuleNotFoundError as error:
    print(f'Missing optional package: {error.name}. Install it with: py -m pip install shap')
else:
    artifact_dir = Path('artifacts')
    classifier = joblib.load(artifact_dir / 'classifier.joblib')['model']
    explanation_data = prepare_data(pd.read_csv(artifact_dir / 'notebook_encoded.csv'))[FEATURES]
    shap_data = explanation_data.sample(min(200, len(explanation_data)), random_state=42).reset_index(drop=True)

    shap_matrix = shap_data.copy()
    category_levels = {}
    for feature in CATEGORICAL:
        levels = shap_data[feature].astype('string').fillna('Unknown').drop_duplicates().tolist()
        category_levels[feature] = levels
        level_to_code = {level: code for code, level in enumerate(levels)}
        shap_matrix[feature] = shap_data[feature].astype('string').fillna('Unknown').map(level_to_code).astype(float)
    for feature in set(FEATURES) - set(CATEGORICAL):
        shap_matrix[feature] = pd.to_numeric(shap_matrix[feature], errors='coerce').fillna(shap_matrix[feature].median())

    def shap_predict(encoded_rows):
        decoded = pd.DataFrame(encoded_rows, columns=FEATURES)
        for feature in FEATURES:
            if feature in category_levels:
                levels = category_levels[feature]
                codes = np.rint(pd.to_numeric(decoded[feature], errors='coerce').fillna(0)).astype(int)
                codes = np.clip(codes, 0, len(levels) - 1)
                decoded[feature] = [levels[code] for code in codes]
            else:
                decoded[feature] = pd.to_numeric(decoded[feature], errors='coerce')
        return classifier.predict_proba(decoded[FEATURES])[:, 1]

    background = shap_matrix.sample(min(25, len(shap_matrix)), random_state=42).to_numpy(dtype=float)
    shap_sample = shap_matrix.iloc[:min(5, len(shap_matrix))].to_numpy(dtype=float)
    shap_explainer = shap.KernelExplainer(shap_predict, background, feature_names=FEATURES)
    shap_values = np.asarray(shap_explainer.shap_values(
        shap_sample, nsamples=2 * len(FEATURES) + 2
    ))

    print('SHAP values explain the calibrated probability of a missed appointment (no_show = 1).')
    shap.summary_plot(shap_values, shap_sample, feature_names=FEATURES, plot_type='bar', max_display=12)
    first_explanation = shap.Explanation(
        values=shap_values[0],
        base_values=shap_explainer.expected_value,
        data=shap_sample[0],
        feature_names=FEATURES,
    )
    shap.plots.waterfall(first_explanation, max_display=12)


In [ ]:
# LIME: explain the calibrated no-show probability for one appointment.
try:
    from pathlib import Path
    import joblib
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from lime.lime_tabular import LimeTabularExplainer
    from clinic_ml import CATEGORICAL, FEATURES, prepare_data
except ModuleNotFoundError as error:
    print(f'Missing optional package: {error.name}. Install it with: py -m pip install lime')
else:
    artifact_dir = Path('artifacts')
    classifier = joblib.load(artifact_dir / 'classifier.joblib')['model']
    explanation_data = prepare_data(pd.read_csv(artifact_dir / 'notebook_encoded.csv'))[FEATURES]
    lime_data = explanation_data.sample(min(500, len(explanation_data)), random_state=42).reset_index(drop=True)

    # LIME needs a positive scale for every continuous field. Remove fields that
    # are constant in this sample; their fixed value is restored for prediction.
    lime_features = [
        feature for feature in FEATURES
        if feature in CATEGORICAL or lime_data[feature].nunique(dropna=False) > 1
    ]
    constant_features = [feature for feature in FEATURES if feature not in lime_features]
    if constant_features:
        print('Excluded constant LIME features:', ', '.join(constant_features))

    # LIME requires numbers. Encode only for LIME, then decode categories before
    # passing perturbed records back to the saved classifier.
    lime_matrix = lime_data[lime_features].copy()
    categorical_indices = []
    categorical_names = {}
    category_levels = {}
    for feature_index, feature in enumerate(lime_features):
        if feature in CATEGORICAL:
            levels = lime_data[feature].astype('string').fillna('Unknown').drop_duplicates().tolist()
            category_levels[feature] = levels
            level_to_code = {level: code for code, level in enumerate(levels)}
            lime_matrix[feature] = lime_data[feature].astype('string').fillna('Unknown').map(level_to_code).astype(float)
            categorical_indices.append(feature_index)
            categorical_names[feature_index] = levels
        else:
            lime_matrix[feature] = pd.to_numeric(lime_data[feature], errors='coerce').fillna(lime_data[feature].median())

    fixed_values = lime_data.iloc[0][FEATURES].copy()

    def lime_predict(encoded_rows):
        decoded = pd.DataFrame([fixed_values.to_dict()] * len(encoded_rows), columns=FEATURES)
        perturbed = pd.DataFrame(encoded_rows, columns=lime_features)
        for feature in lime_features:
            if feature in category_levels:
                levels = category_levels[feature]
                codes = np.rint(pd.to_numeric(perturbed[feature], errors='coerce').fillna(0)).astype(int)
                codes = np.clip(codes, 0, len(levels) - 1)
                decoded[feature] = [levels[code] for code in codes]
            else:
                decoded[feature] = pd.to_numeric(perturbed[feature], errors='coerce')
        return classifier.predict_proba(decoded[FEATURES])

    lime_explainer = LimeTabularExplainer(
        training_data=lime_matrix.to_numpy(dtype=float),
        feature_names=lime_features,
        class_names=['attended', 'missed'],
        categorical_features=categorical_indices,
        categorical_names=categorical_names,
        mode='classification',
        discretize_continuous=True,
        random_state=42,
    )
    appointment_index = 0
    lime_explanation = lime_explainer.explain_instance(
        lime_matrix.iloc[appointment_index].to_numpy(dtype=float),
        lime_predict,
        labels=(1,),
        num_features=min(12, len(lime_features)),
    )
    print(f'LIME explanation for cleaned appointment row {appointment_index}; class: missed appointment.')
    lime_explanation.as_pyplot_figure(label=1)
    plt.tight_layout()
    plt.show()


## Feature selection recommendation

Use this global permutation-importance result to decide which predictor columns to retain or consider removing. This is more reliable for feature selection than one local LIME explanation. **Retain** means the feature has a stable ROC-AUC contribution; **review** means it has a small but positive contribution; **candidate to remove** means its measured contribution is zero, negative, or smaller than its uncertainty. Retrain the model after removing any candidate and compare the held-out test metrics before making the deletion permanent.


In [ ]:
# Recommend which predictor columns to retain, review, or consider removing.
from pathlib import Path
import numpy as np
import pandas as pd

importance_path = Path('artifacts') / 'feature_importance.csv'
if not importance_path.exists():
    print('Feature importance is unavailable. Run: py train_models.py')
else:
    feature_selection = pd.read_csv(importance_path)
    feature_selection['importance_lower_bound'] = (
        feature_selection['auc_decrease'] - feature_selection['std']
    )

    def recommendation(row):
        if row['importance_lower_bound'] >= 0.001:
            return 'Retain'
        if row['importance_lower_bound'] > 0:
            return 'Review'
        return 'Candidate to remove'

    feature_selection['recommendation'] = feature_selection.apply(recommendation, axis=1)
    feature_selection = feature_selection.sort_values(
        ['recommendation', 'auc_decrease'],
        key=lambda values: values.map({
            'Retain': 0, 'Review': 1, 'Candidate to remove': 2
        }).fillna(values),
    ).reset_index(drop=True)

    display(feature_selection[[
        'feature', 'auc_decrease', 'std', 'importance_lower_bound', 'recommendation'
    ]].round(5))

    for label in ['Retain', 'Review', 'Candidate to remove']:
        columns = feature_selection.loc[
            feature_selection['recommendation'].eq(label), 'feature'
        ].tolist()
        print(f'{label}: {", ".join(columns) or "none"}')


## Apply Review and Candidate-to-remove recommendations

This final step removes the columns marked **Review** or **Candidate to remove** by the preceding global permutation-importance analysis from the current notebook data and encoded training artifact. The model configuration also excludes these fields, so rerun `py train_models.py` after this notebook step to train the reduced-feature model.


In [ ]:
# Remove the current Review and Candidate-to-remove columns from notebook data and the encoded artifact.
from pathlib import Path
import pandas as pd

if 'feature_selection' not in globals():
    raise RuntimeError('Run the Feature selection recommendation section first.')

columns_to_remove = feature_selection.loc[
    feature_selection['recommendation'].isin(['Review', 'Candidate to remove']), 'feature'
].tolist()
print('Removing Review and Candidate-to-remove columns:', ', '.join(columns_to_remove) or 'none')

if 'df' in globals():
    df = df.drop(columns=columns_to_remove, errors='ignore')
    df.to_csv(Path('artifacts') / 'notebook_cleaned.csv', index=False)

if 'encoded_data' in globals():
    encoded_data = encoded_data.drop(columns=columns_to_remove, errors='ignore')
else:
    encoded_path = Path('artifacts') / 'notebook_encoded.csv'
    encoded_data = pd.read_csv(encoded_path).drop(columns=columns_to_remove, errors='ignore')

encoded_path = Path('artifacts') / 'notebook_encoded.csv'
encoded_data.to_csv(encoded_path, index=False)
print(f'Saved reduced encoded data: {encoded_data.shape[0]:,} rows x {encoded_data.shape[1]:,} columns')
